In [18]:
import numpy as np                              
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.utils import check_random_state
from sklearn.decomposition import TruncatedSVD
from collections import defaultdict
from sklearn.utils import check_random_state

In [20]:
# Reading in the relevant datasets
ratings_prepared_data = pd.read_csv('data/df_ratings_prepared.csv')
books_prepared_data = pd.read_csv('data/df_books_prepared.csv')

In [21]:
# Peak into books data
books_prepared_data.head()

,isbn,book_title,book_author,publisher,year_of_publication,_title_norm,_author_norm,_publisher_norm,_decade_tok
0,0195153448,Classical Mythology,Mark P. O. Morford,Oxford University Press,2002.0,classical mythology,mark pomorford,oxford university press,decade_2000
1,0002005018,Clara Callan,Richard Bruce Wright,HarperFlamingo Canada,2001.0,clara callan,richard bruce wright,harperflamingo canada,decade_2000
2,0060973129,Decision in Normandy,Carlo D'Este,HarperPerennial,1991.0,decision in normandy,carlo d'este,harperperennial,decade_1990
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,Farrar Straus Giroux,1999.0,flu: the story of the great influenza pandemic...,gina bari kolata,farrar straus giroux,decade_1990
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,W. W. Norton &amp; Company,1999.0,the mummies of urumchi,ejwbarber,w. w. norton & company,decade_1990


In [22]:
# Shape of data
ratings_prepared_data.shape

(1009521, 3)

In [23]:
# Peak into ratings data
ratings_prepared_data.head()

,user,isbn,rating
0,276725,034545104X,0
1,2313,034545104X,5
2,6543,034545104X,0
3,8680,034545104X,5
4,10314,034545104X,9


In [28]:
print(f"Number of rows in Ratings data is: {ratings_prepared_data.shape[0]}")
print('-'*50)
print(f"Number of unique users in Ratings data is: {len(ratings_prepared_data['user'].unique())}")
print(f"Number of unique ISBNs in Ratings data is: {len(ratings_prepared_data['isbn'].unique())}")

Number of rows in Ratings data is: 1009521
--------------------------------------------------
Number of unique users in Ratings data is: 90381
Number of unique ISBNs in Ratings data is: 262793


In [30]:
groupby_user_to_isbn = ratings_prepared_data.groupby('user')['isbn'].count().sort_values(ascending=False).rename('count').reset_index()
groupby_user_to_isbn.head()

,user,count
0,11676,10837
1,198711,6297
2,98391,5777
3,153662,5775
4,35859,5591


In [32]:
groupby_isbn_to_user = ratings_prepared_data.groupby('isbn')['user'].count().sort_values(ascending=False).rename('count').reset_index()
groupby_isbn_to_user.head()

,isbn,count
0,0971880107,2502
1,0316666343,1295
2,0385504209,883
3,0060928336,732
4,0312195516,723


# Preparation of Data:
Users/items with only one rating contribute almost no collaborative signal (no co-occurrence to compute similarity). For Collaborative Filtering training, it’s standard to drop users/items below a small minimum.

In [35]:
# Explore thresholds vs retained size/density
def retention_after_filter(df, min_users, min_isbns):
    g_users = df.groupby("user")["isbn"].count()
    g_isbns = df.groupby("isbn")["user"].count()
    df_users = df[df["user"].isin(g_users[g_users >= min_users].index)]
    g_isbns_threshed = df_users.groupby("isbn")["user"].count()
    df_isbns = df_users[df_users["isbn"].isin(g_isbns_threshed[g_isbns_threshed >= min_isbns].index)]
    n_users = df_users["user"].nunique()
    n_items = df_isbns["isbn"].nunique()
    density = len(df_isbns) / (max(n_users,1) * max(n_items,1))
    return len(df_isbns), n_users, n_items, density

In [37]:
candidates = [(3,3), (5,5), (10,5), (5,10), (10,10)]

retention_summary = list()

for max_u, max_i in candidates:
    kept, users, isbns, density = retention_after_filter(ratings_prepared_data, max_u, max_i)
    retention_summary.append({"min_users": max_u, 
                              "min_isbns": max_i, 
                              "rows_kept": kept, 
                              "users": users, 
                              "isbns": isbns, 
                              "density": density, 
                              "rows_lost":ratings_prepared_data.shape[0]-kept})
pd.DataFrame(retention_summary).sort_values(["min_users","min_isbns"])

,min_users,min_isbns,rows_kept,users,isbns,density,rows_lost
0,3,3,708762,29560,69904,0.000343,300759
1,5,5,573665,20069,36419,0.000785,435856
3,5,10,439074,20069,15367,0.001424,570447
2,10,5,528741,11556,34525,0.001325,480780
4,10,10,399335,11556,14328,0.002412,610186


- Lets consider the first two pairs of candidates (3, 3) and (5, 5) for `MIN_USER_INTERACTIONS` and `MIN_ITEM_INTERACTIONS` 

In [40]:
MIN_USER_INTERACTIONS = 3
MIN_ISBN_INTERACTIONS = 3

In [42]:
g_users = ratings_prepared_data.groupby("user")["isbn"].count()
g_isbns = ratings_prepared_data.groupby("isbn")["user"].count()

In [44]:
df_3u = ratings_prepared_data[ratings_prepared_data["user"].isin(g_users[g_users >= MIN_USER_INTERACTIONS].index)]
g_isbns = df_3u.groupby("isbn")["user"].count()
df_3u3i = df_3u[df_3u["isbn"].isin(g_isbns[g_isbns >= MIN_ISBN_INTERACTIONS].index)]

print("Before:", ratings_prepared_data.shape)
print("After:", df_3u3i.shape)
print("Users:", ratings_prepared_data['user'].nunique(), " to ", df_3u3i['user'].nunique())
print("Items:", ratings_prepared_data['isbn'].nunique(), " to ", df_3u3i['isbn'].nunique())

Before: (1009521, 3)
After: (708762, 3)
Users: 90381  to  29007
Items: 262793  to  69904


In [46]:
RNG_SEED = 42
rng = check_random_state(RNG_SEED)

In [48]:
def train_test_split_per_user(df, test_frac=0.2, rng=None):
    if rng is None:
        rng = np.random.default_rng(42)
    df = df.sort_values(["user", "isbn"]).copy()
    test_idx = []
    for u, block in df.groupby("user"):
        idx = block.index.to_numpy()
        n_test = max(1, int(round(test_frac * len(idx))))
        test_sel = rng.choice(idx, size=n_test, replace=False)
        test_idx.extend(test_sel.tolist())
    mask = df.index.isin(test_idx)
    test = df.loc[mask].copy()
    train = df.loc[~mask].copy()
    return train, test

In [50]:
train, test = train_test_split_per_user(df_3u3i, test_frac=0.2, rng=rng)
print(f"Train's Shape: {train.shape}")
print(f"Test's Shape: {test.shape}")

Train's Shape: (562234, 3)
Test's Shape: (146528, 3)


In [52]:
POS_THRESH = 7 #train["rating"].median()

train_pos = train[train["rating"] >= POS_THRESH].copy()
test_pos  = test[test["rating"]  >= POS_THRESH].copy()

print("Total Train (train): ", len(train),
      "\nTotal Test (test):", len(test),
      "\n","-"*40,
      "\nPositive threshold:", POS_THRESH, 
      "\n","-"*40,
      "\nTrain positives (train_pos):", len(train_pos), 
      "\nTest positives (test_pos):", len(test_pos))

Total Train (train):  562234 
Total Test (test): 146528 
 ---------------------------------------- 
Positive threshold: 7 
 ---------------------------------------- 
Train positives (train_pos): 148926 
Test positives (test_pos): 40012


In [54]:
users = pd.Index(train["user"].unique(), name="user")
isbns = pd.Index(train["isbn"].unique(), name="isbn")

u2i, i2i = dict(), dict()

for i, val in enumerate(users):
    u2i[val] = i

for i, val in enumerate(isbns):
    i2i[val] = i

In [60]:
def to_csr(df_like):
    rows = df_like["user"].map(u2i, na_action="ignore").dropna().astype(int)
    cols = df_like["isbn"].map(i2i, na_action="ignore").dropna().astype(int)
    data = df_like["rating"].reindex(rows.index).to_numpy()
    return csr_matrix((data, (rows, cols)), shape=(len(users), len(isbns)))

# For implicit (binary) version too:
def to_csr_binary(df_like):
    rows = df_like["user"].map(u2i, na_action="ignore").dropna().astype(int)
    cols = df_like["isbn"].map(i2i, na_action="ignore").dropna().astype(int)
    data = np.ones(len(rows), dtype=np.float32)
    return csr_matrix((data, (rows, cols)), shape=(len(users), len(isbns)))

In [62]:
R = to_csr(train)                # explicit ratings
R_bin = to_csr_binary(train_pos) # implicit positives
print(f"R's shape: {R.shape} | R_bin's shape: {R_bin.shape}")

R's shape: (27774, 69687) | R_bin's shape: (27774, 69687)


### Popularity baseline

In [84]:
isbn_pop = train_pos.groupby("isbn")["user"].nunique().sort_values(ascending=False)
pop_isbns = item_pop.index.to_numpy()

print(len(pop_items))

45161


In [86]:
def recommend_popularity(user_id_str, k=10):
    # seen items in train
    seen = set(train[train["user"] == user_id_str]["isbn"])
    recs = [i for i in pop_items if i not in seen][:k]
    return recs

In [88]:
# We fit on item-user matrix (transpose) for item neighbors
knn_k = 50  # number of neighbors per item
isbn_user = R_bin.T  # shape: (n_items, n_users)

knn = NearestNeighbors(metric="cosine", algorithm="auto", n_neighbors=min(knn_k, isbn_user.shape[0]-1))
knn.fit(isbn_user)

# Precompute neighbors for all items for speed
# distances: cosine distance; similarity = 1 - distance
distances, neighbors = knn.kneighbors(isbn_user, n_neighbors=min(knn_k, isbn_user.shape[0]))
isbn_neighbors = neighbors
isbn_sims = 1 - distances
print(isbn_neighbors.shape, isbn_sims.shape)

(69687, 50) (69687, 50)


In [90]:
def user_positive_isbns(user_id_str):
    return set(train_pos.loc[train_pos["user"] == user_id_str, "isbn"])

def recommend_itemknn(user_id_str, k=10, aggregate="sum"):
    pos_items = list(user_positive_isbns(user_id_str))
    if not pos_items:
        # cold or no positives -> fallback to popularity
        return recommend_popularity(user_id_str, k=k)
    
    # candidate scores
    scores = defaultdict(float)
    for it in pos_items:
        if it not in i2i: 
            continue
        j = i2i[it]
        nbrs = isbn_neighbors[j]
        sims = isbn_sims[j]
        for neighbor_idx, sim in zip(nbrs, sims):
            if neighbor_idx == j or sim <= 0:
                continue
            candidate_isbn = isbns[neighbor_idx]
            scores[candidate_isbn] += sim  # "sum" aggregator
    
    # remove seen items (all train interactions)
    seen = set(train.loc[train["user"] == user_id_str, "isbn"])
    ranked = [it for it, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True) if it not in seen]
    if len(ranked) < k:
        # top up with popularity
        ranked = ranked + [i for i in pop_isbns if i not in set(ranked)|seen]
    return ranked[:k]


In [99]:
# Compute user means on train; subtract to reduce global/user bias
user_means = np.zeros(len(users), dtype=np.float32)
R_csr = R.tocsr(copy=True)  # explicit
for u_idx in range(R_csr.shape[0]):
    start, end = R_csr.indptr[u_idx], R_csr.indptr[u_idx+1]
    if end > start:
        vals = R_csr.data[start:end]
        mu = vals.mean()
        R_csr.data[start:end] = vals - mu
        user_means[u_idx] = mu

rank = 64
svd = TruncatedSVD(n_components=rank, random_state=RNG_SEED)
U = svd.fit_transform(R_csr)           # (n_users, rank)
S = svd.singular_values_               # (rank,)
Vt = svd.components_                   # (rank, n_items)

# function to score all items for a single user
def svd_scores_for_user(user_id_str):
    if user_id_str not in u2i:
        # cold user -> use popularity scores
        return pd.Series(index=isbns, data=np.nan)
    u_idx = u2i[user_id_str]
    pred_centered = U[u_idx] @ Vt   # (n_items,)
    pred = pred_centered + user_means[u_idx]
    return pd.Series(pred, index=isbns)

def recommend_svd(user_id_str, k=10):
    s = svd_scores_for_user(user_id_str)
    # remove seen items
    seen = set(train.loc[train["user"] == user_id_str, "isbn"])
    s = s.drop(index=list(seen), errors="ignore")
    topk = s.sort_values(ascending=False).index[:k].tolist()
    if len(topk) < k:
        topk += [i for i in pop_isbns if i not in set(topk)|seen][:k-len(topk)]
    return topk


In [105]:
test_pos_by_user = test_pos.groupby("user")["isbn"].apply(set).to_dict()
candidate_users = list(test_pos_by_user.keys())

def precision_recall_at_k(recommender_fn, users_list, k=10):
    tp, total_pred, total_true = 0, 0, 0
    for u in users_list:
        truth = test_pos_by_user.get(u, set())
        preds = recommender_fn(u, k=k)
        if isinstance(preds, list):
            preds = preds[:k]
        else:
            preds = list(preds)[:k]
        hits = len(set(preds) & truth)
        tp += hits
        total_pred += k
        total_true += len(truth)
    precision = tp / total_pred if total_pred else 0.0
    recall = tp / total_true if total_true else 0.0
    return precision, recall

def coverage_at_k(recommender_fn, users_list, k=10):
    rec_items = set()
    for u in users_list:
        rec_items.update(recommender_fn(u, k=k))
    return len(rec_items) / len(isbns)

for K in [5, 10, 20]:
    p_pop, r_pop = precision_recall_at_k(recommend_popularity, candidate_users, k=K)
    p_knn, r_knn = precision_recall_at_k(recommend_itemknn, candidate_users, k=K)
    p_svd, r_svd = precision_recall_at_k(recommend_svd, candidate_users, k=K)

    cov_pop = coverage_at_k(recommend_popularity, candidate_users, k=K)
    cov_knn = coverage_at_k(recommend_itemknn, candidate_users, k=K)
    cov_svd = coverage_at_k(recommend_svd, candidate_users, k=K)

    print(f"K={K:>2} | POP:  P={p_pop:.4f} R={r_pop:.4f} Cov={cov_pop:.3f}  | "
          f"kNN: P={p_knn:.4f} R={r_knn:.4f} Cov={cov_knn:.3f}  | "
          f"SVD: P={p_svd:.4f} R={r_svd:.4f} Cov={cov_svd:.3f}")


K= 5 | POP:  P=0.0044 R=0.0087 Cov=0.000  | kNN: P=0.0068 R=0.0133 Cov=0.290  | SVD: P=0.0047 R=0.0093 Cov=0.016
K=10 | POP:  P=0.0035 R=0.0139 Cov=0.000  | kNN: P=0.0042 R=0.0165 Cov=0.377  | SVD: P=0.0036 R=0.0140 Cov=0.030
K=20 | POP:  P=0.0029 R=0.0224 Cov=0.001  | kNN: P=0.0027 R=0.0212 Cov=0.477  | SVD: P=0.0027 R=0.0211 Cov=0.052


In [107]:
some_user = train["user"].iloc[0]
print("User:", some_user)

print("\nPopularity: ", recommend_popularity(some_user, k=10))
print("\nItem-kNN: ", recommend_itemknn(some_user, k=10))
print("\nSVD: ", recommend_svd(some_user, k=10))


User: 8

Popularity:  ['0316666343', '0385504209', '059035342X', '0312195516', '0142001740', '0671027360', '0446672211', '0060928336', '0452282152', '043935806X']

Item-kNN:  ['078686043X', '067101756X', '0553252275', '0875964125', '0671798057', '0451173392', '0425132889', '0679883886', '078688097X', '0836204255']

SVD:  ['0446672211', '0142001740', '0446310786', '0385504209', '067976402X', '0345337662', '059035342X', '0312966970', '1558531025', '0553289411']
